# Governance Logging Demo

Build order step 7 (`CLAUDE.md`): persist the full audit trail of every boardroom session,
not just produce a validated `BoardRecommendation` and let it evaporate once `BossAgent.run()`
returns.

## Why this is a separate layer, not baked into `BossAgent`

Every specialist agent already logs its own tool calls automatically — `SpecialistAgent._call_tool()`
(`backend/agents/base_agent.py`) captures a `ToolCallRecord` for every tool invocation,
unconditionally, at the per-tool level. That's per-*tool-call* traceability, already live since
the very first agent was built.

What's new here is the layer above it: one `GovernanceLog` per **session** (a full boss-agent
run), capturing the query, the complete `BoardRecommendation`, which model versions produced
it, total execution time, and — filled in later — a human's accept/reject/modify decision.

This is deliberately a wrapper *around* `BossAgent`, not a method added to it —
`GovernanceLogger` (`backend/governance/logger.py`) takes a `BossAgent` instance and a
`MongoClient` instance, and composes them. `BossAgent` stays a pure function of
`query -> BoardRecommendation`, exactly like `DatabricksClient` is the only file that knows SQL
and `BossAgent` doesn't know Databricks exists at the boss-agent level. Same separation of
concerns, applied one layer up.

## Storage: MongoDB Atlas (free tier)

CLAUDE.md's stack table names MongoDB/Mongoose for session storage. This uses **MongoDB
Atlas's M0 free tier** — a genuinely free, permanent tier (512MB, no trial expiry), consistent
with the cost-conscious choices already made for the boss LLM (Groq) and the RAG layer (local
embeddings). `backend/governance/mongo_client.py` is the only file that imports `pymongo`
directly — same isolation pattern as `DatabricksClient`.

## What gets logged, and why each field matters

| Field | Captures | Why (CLAUDE.md section 6) |
|---|---|---|
| `session_id` | UUID per session | Lets a human decision get attached to the right session after the fact |
| `user_query` | The original question | Traceability — what prompted this recommendation |
| `recommendation` | The full validated `BoardRecommendation` (briefings, dissents, synthesis, confidence, action items) | The complete decision-support output, not just a summary |
| `model_versions` | `{boss_llm, embedding_model}` | "Model/version logging per session for explainability across model swaps" — if GPT OSS 20B gets swapped for Llama 3.3 70B later, old sessions stay attributable to the model that actually produced them |
| `total_execution_time_ms` | Wall-clock time for the full session | Operational visibility |
| `human_decision` / `human_notes` | Filled in later via `record_human_decision()` | Closes the "Human Oversight" pillar loop — a recommendation isn't a decision until a human actually accepts, rejects, or modifies it |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.boss import BossAgent
from backend.db import DatabricksClient
from backend.governance import GovernanceLogger, MongoClient

db = DatabricksClient()
boss = BossAgent(db)
mongo = MongoClient()

mongo.ping()
print("MongoDB Atlas connection OK")

logger = GovernanceLogger(boss, mongo)

MongoDB Atlas connection OK


## Run a session with logging

In [2]:
log = logger.run_with_logging("How is our financial health looking?")

print(f"Session ID: {log.session_id}")
print(f"Agents invoked: {[a.value for a in log.recommendation.agents_invoked]}")
print(f"Model versions: {log.model_versions}")
print(f"Total execution time: {log.total_execution_time_ms:.0f}ms")
print(f"Human decision: {log.human_decision}  (not yet reviewed)")
print(f"\nSynthesis:\n{log.recommendation.synthesis[:400]}...")

Session ID: 75732b8a-c87f-44ce-a659-a4df67acecbc
Agents invoked: ['Finance']
Model versions: {'boss_llm': 'openai/gpt-oss-20b', 'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2'}
Total execution time: 20224ms
Human decision: None  (not yet reviewed)

Synthesis:
Board Memo – Financial Health Overview

1. **Contribution Margin** – Finance's calculate_margin_trend tool reports an improving trend over the last 12 months, indicating better cost control or pricing.

2. **Revenue Anomalies** – Finance's detect_revenue_anomalies tool flagged 13 anomalous daily revenue points out of 614 days, a warning that requires investigation to rule out reporting or operatio...


## Verify it actually persisted

Fetch the same session back from MongoDB by ID, independent of the in-memory `log` object
above — this proves it's really in the database, not just returned in-process.

In [3]:
fetched = logger.get_session(log.session_id)
assert fetched is not None, "session not found in MongoDB!"
assert fetched.user_query == log.user_query
print(f"Fetched session {fetched.session_id} from MongoDB - matches: {fetched.user_query == log.user_query}")

Fetched session 75732b8a-c87f-44ce-a659-a4df67acecbc from MongoDB - matches: True


## Close the human-oversight loop

`BoardRecommendation.requires_human_approval` has defaulted to `True` since the schema was
first written — this is the step that actually exercises it. A recommendation isn't a decision
until a human accepts, rejects, or modifies it.

In [4]:
logger.record_human_decision(
    log.session_id,
    decision="accepted",
    notes="Reviewed - COGS gap needs a data source, otherwise consistent with what we already knew.",
)

updated = logger.get_session(log.session_id)
print(f"Human decision: {updated.human_decision}")
print(f"Human notes: {updated.human_notes}")

Human decision: accepted
Human notes: Reviewed - COGS gap needs a data source, otherwise consistent with what we already knew.


## Run a second session and list recent sessions

In [5]:
logger.run_with_logging("Are customers happy with delivery performance?")

recent = logger.list_recent_sessions(limit=5)
for s in recent:
    decision = s.human_decision or "pending review"
    print(f"[{s.timestamp.isoformat()[:19]}] {s.user_query[:60]!r} -> {[a.value for a in s.recommendation.agents_invoked]} ({decision})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[2026-07-16T14:04:57] 'Are customers happy with delivery performance?' -> ['Operations', 'Sentiment'] (pending review)
[2026-07-16T14:04:26] 'How is our financial health looking?' -> ['Finance'] (accepted)


## Governance completeness - the audit trail is queryable

This is the data foundation the eval suite (build order step 9) will check against:
"Governance completeness — every session log has a full audit trail (pass/fail, structural)".
That check can now run for real, against real persisted sessions, not a hypothetical schema.

In [6]:
log_dict = fetched.model_dump()
required_fields = ["session_id", "user_query", "recommendation", "model_versions", "timestamp"]
missing = [f for f in required_fields if log_dict.get(f) is None]
print(f"Audit trail complete: {len(missing) == 0}" + (f" (missing: {missing})" if missing else ""))

Audit trail complete: True


## What's still pending

Per the build order in `CLAUDE.md`:

8. MERN frontend + streaming + WebSocket agent status
9. Eval suite (governance-completeness checks now have real data to run against)
10. Deployment containerization